In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents[10]

{'id': '310a39c0f9',
 'course': 'ai-dev-tools-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Can I take the course on a locked-down work laptop without admin rights?',
 'answer': 'Yes. Use GitHub Codespaces - you only need a modern browser (or VS Code) and an internet connection, and you can install whatever you need inside the Codespace. GitHub Copilot and OpenAI Codex both work in Codespaces. The free tier is usually generous enough for the course.'}

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

153

In [4]:
documents = documents_llm

In [5]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

    # this creates structured information (which means the llm will produce outputs in structured form, the reaoson this is important is that we can then parse the output and use it in our code) 

In [7]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
anthropic_client = Anthropic()

In [9]:
import json

user_prompt = json.dumps(doc)

In [10]:
messages=[
            {
                "role": "user",
                "content": user_prompt,
            }
        ]

In [11]:
response = anthropic_client.beta.messages.parse(
    model="claude-haiku-4-5-20251001",
    max_tokens=2048,
    betas=["structured-outputs-2025-11-13"],
    messages=messages,
    output_format=Questions
)

In [12]:
result = response.parsed_output

print(result)

questions=['I just discovered the course. Can I still join?', 'What are the requirements to receive a certificate?', 'When is the deadline for project submissions?', 'Can I join the course after the official start date?', 'Is there a late enrollment option available?']


In [13]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [14]:
print(result.questions)

['I just discovered the course. Can I still join?', 'What are the requirements to receive a certificate?', 'When is the deadline for project submissions?', 'Can I join the course after the official start date?', 'Is there a late enrollment option available?']


In [15]:
from evaluation_utils import llm_structured, calc_price

In [16]:
result, usage = llm_structured(
    anthropic_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

/tmp/ipykernel_1893/3895614265.py:1: DeprecationWarning: The 'output_format' parameter is deprecated. Please use 'output_config.format' instead.
  result, usage = llm_structured(


['Is it too late to enroll in this course if I just found out about it?', 'Do I need to join at a specific time to get the certificate, or can I start anytime?', 'What happens if I sign up now but finish the project after submissions close?', 'Can I participate in the course without completing the project requirements?', 'Is there a deadline for submitting the final project to qualify for certification?']


In [17]:
usage

BetaUsage(cache_creation=BetaCacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=354, iterations=None, output_tokens=97, output_tokens_details=None, server_tool_use=None, service_tier='standard', speed=None)

In [18]:
calc_price(usage)

{'input_cost': 0.000354,
 'output_cost': 0.00048499999999999997,
 'total_cost': 0.0008389999999999999}

In [19]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it too late to enroll in this course if I just found out about it?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to join at a specific time to get the certificate, or can I start anytime?',
  'document': '74eb249bbf'},
 {'question': 'What happens if I sign up now but finish the project after submissions close?',
  'document': '74eb249bbf'},
 {'question': 'Can I participate in the course without completing the project requirements?',
  'document': '74eb249bbf'},
 {'question': 'Is there a deadline for submitting the final project to qualify for certification?',
  'document': '74eb249bbf'}]

In [20]:
import pandas as pd

In [22]:
pd.DataFrame(records)

,question,document
0,Is it too late to enroll in this course if I j...,74eb249bbf
1,Do I need to join at a specific time to get th...,74eb249bbf
2,What happens if I sign up now but finish the p...,74eb249bbf
3,Can I participate in the course without comple...,74eb249bbf
4,Is there a deadline for submitting the final p...,74eb249bbf


In [23]:
from evaluation_utils import llm_structured_retry

In [28]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        anthropic_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [29]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [30]:
generate_ground_truth(doc)

([{'question': "How do I find myself on the leaderboard if I can't see my name?",
   'document': 'c2903069a0'},
  {'question': "What's that random name assigned to my account and can I change it?",
   'document': 'c2903069a0'},
  {'question': 'Where can I check what display name shows up for me in the course?',
   'document': 'c2903069a0'},
  {'question': 'Does my leaderboard name have to be my real name or can I use a nickname?',
   'document': 'c2903069a0'},
  {'question': 'What name will actually appear on my certificate at the end?',
   'document': 'c2903069a0'}],
 BetaUsage(cache_creation=BetaCacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=550, iterations=None, output_tokens=95, output_tokens_details=None, server_tool_use=None, service_tier='standard', speed=None))

In [31]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [32]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/153 [00:00<?, ?it/s]

In [33]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

765

In [35]:
results[0]

([{'question': "Can I enroll in this course if I'm just finding out about it now?",
   'document': '74eb249bbf'},
  {'question': "What's the deadline for submitting my project if I want to get a certificate?",
   'document': '74eb249bbf'},
  {'question': 'Do I need to start the course at a specific time or can I join whenever I want?',
   'document': '74eb249bbf'},
  {'question': 'If I join late, am I still able to complete everything and get certified?',
   'document': '74eb249bbf'},
  {'question': 'Is there a cutoff date for when I have to turn in my final project to receive the certificate?',
   'document': '74eb249bbf'}],
 BetaUsage(cache_creation=BetaCacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=354, iterations=None, output_tokens=106, output_tokens_details=None, server_tool_use=None, service_tier='standard', speed=None))

In [36]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.1659139999999999

In [37]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.1659139999999999

In [38]:
df_ground_truth = pd.DataFrame(ground_truth)

In [40]:
df_ground_truth

,question,document
0,Can I enroll in this course if I'm just findin...,74eb249bbf
1,What's the deadline for submitting my project ...,74eb249bbf
2,Do I need to start the course at a specific ti...,74eb249bbf
3,"If I join late, am I still able to complete ev...",74eb249bbf
4,Is there a cutoff date for when I have to turn...,74eb249bbf
...,...,...
760,What do I need to change to use Groq as my LLM...,54b98d3581
761,How do I set up Groq in the agent.py file and ...,54b98d3581
762,Do I need to modify the SearchDeps and instruc...,54b98d3581
763,What's the proper model string format for usin...,54b98d3581


In [39]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

OSError: Cannot save file into a non-existent directory: 'data'